# FINRL Walk-Forward Experiment

This notebook runs the Phase 12 walk-forward experiment runner and visualizes portfolio performance against the S&P 500 / SPY benchmark, plus spectral feature evolution over time.

Use small synthetic data locally. Use Colab for the full configured universe, encoder training, PPO training, and full walk-forward experiments.

In [ ]:
# Colab setup. Uncomment after cloning the repository in Colab.
# %pip install -e .

from datetime import date, timedelta

import polars as pl

from finrl.backtest.walk_forward import WalkForwardConfig
from finrl.experiments import (
    ExperimentConfig,
    RawExperimentData,
    build_performance_figure,
    build_spectral_figure,
    metrics_to_frame,
    run_walk_forward_experiment,
)
from finrl.features.preprocessing import PreprocessingConfig
from finrl.features.schema import FeatureBundle
from finrl.models.encoder import EncoderConfig
from finrl.ppo.policy import PPOConfig
from finrl.regimes.schema import HMMConfig

## Prepared Data Contract

The runner expects prepared feature and return tables:

- `FeatureBundle` with asset, macro, and 20 spectral feature columns.
- `returns`: Polars DataFrame with `decision_date` and one return column per tradable asset, including cash.
- `spy_returns`: Polars DataFrame with `decision_date` and `spy_return` for the same holding periods.

Replace the synthetic fixture below with the output of the data, feature, preprocessing, and return-preparation pipeline for full experiments.

In [ ]:
def make_synthetic_data() -> RawExperimentData:
    dates = []
    for year in (2020, 2021, 2022):
        start = date(year, 1, 3)
        dates.extend(start + timedelta(days=7 * index) for index in range(8))
    dates = tuple(dates)
    tickers = ("AAA", "BBB")

    asset_rows = []
    for day_index, day in enumerate(dates):
        for ticker_index, ticker in enumerate(tickers):
            asset_rows.append({
                "date": day,
                "ticker": ticker,
                "asset_return": 0.01 * (day_index + 1 + ticker_index),
                "asset_rank": 0.25 + 0.5 * ticker_index,
            })
    asset = pl.DataFrame(asset_rows).with_columns(pl.col("date").cast(pl.Date))
    macro = pl.DataFrame({
        "date": dates,
        "macro_rate": [0.001 * index for index in range(len(dates))],
    }).with_columns(pl.col("date").cast(pl.Date))

    spectral_columns = tuple(f"spectral_{index}" for index in range(20))
    spectral = pl.DataFrame([
        {
            "date": day,
            **{column: float(day_index + column_index / 100.0) for column_index, column in enumerate(spectral_columns)},
        }
        for day_index, day in enumerate(dates)
    ]).with_columns(pl.col("date").cast(pl.Date))

    features = FeatureBundle(
        asset_features=asset,
        macro_features=macro,
        spectral_features=spectral,
        decision_dates=dates,
        tickers=tickers,
        asset_feature_columns=("asset_return", "asset_rank"),
        macro_feature_columns=("macro_rate",),
        spectral_feature_columns=spectral_columns,
    )
    returns = pl.DataFrame({
        "decision_date": dates,
        "AAA": [0.002 + 0.0001 * index for index in range(len(dates))],
        "BBB": [0.001 - 0.00005 * index for index in range(len(dates))],
        "CASH": [0.0001] * len(dates),
    }).with_columns(pl.col("decision_date").cast(pl.Date))
    spy_returns = pl.DataFrame({
        "decision_date": dates,
        "spy_return": [0.0015 + 0.00002 * index for index in range(len(dates))],
    }).with_columns(pl.col("decision_date").cast(pl.Date))
    return RawExperimentData(features=features, returns=returns, spy_returns=spy_returns)

raw_data = make_synthetic_data()

In [ ]:
config = ExperimentConfig(
    walk_forward=WalkForwardConfig(train_years=1, test_years=1, step_years=1),
    preprocessing=PreprocessingConfig(rolling_window=2),
    encoder=EncoderConfig(
        lookback=3,
        n_assets=2,
        asset_feature_dim=2,
        macro_feature_dim=1,
        spectral_feature_dim=20,
    ),
    hmm=HMMConfig(n_states=2, max_iter=5),
    ppo=PPOConfig(n_assets=3, train_epochs=1, learning_rate=1e-4),
    enable_ppo=False,  # switch to True for PPO smoke runs; use Colab for full training
    seed=7,
    periods_per_year=52,
)

result = run_walk_forward_experiment(raw_data, config)
metrics_to_frame(result)

## Performance vs S&P 500

In [ ]:
performance_fig = build_performance_figure(result)
performance_fig.show()

## Spectral Feature Evolution

In [ ]:
spectral_fig = build_spectral_figure(result)
spectral_fig.show()

In [ ]:
# Optional report export
# from finrl.experiments import write_report
# write_report(result, "walk_forward_report")